# Titanic Historical Record Linkage

**Cross-dataset identity matching and verifiable historical outcome lookup**

**Author / project lead:** [@aiaryskaliden](https://github.com/aiaryskaliden)  
**Repository:** [GitHub](https://github.com/aiaryskaliden/titanic-historical-record-linkage)  
**Original executed Kaggle run:** [Kaggle Notebook](https://www.kaggle.com/code/kalidenaiarys/titanic-historical-record-linkage-1309-verified)

> **Disclosure:** This is a deterministic record-linkage project, **not** a machine-learning survival predictor. The Vanderbilt Titanic3 dataset provides published historical outcomes, including those for Kaggle test passengers. A perfect leaderboard result based on these labels must not be interpreted as model accuracy.

## Research question

Can we auditably link every passenger from Kaggle's Titanic competition to an existing historical passenger record, without uncertain fuzzy matches?

## Sources

- [Kaggle Titanic training and test datasets](https://www.kaggle.com/competitions/titanic/data)
- [Vanderbilt Titanic3 CSV](https://hbiostat.org/data/repo/titanic3.csv)
- [Vanderbilt documentation](https://hbiostat.org/data/repo/titanic)

The original files should be downloaded from their publishers. Local execution expects a `data/` folder with `train.csv`, `test.csv`, and `titanic3.csv`. A Kaggle execution can instead use the attached Titanic competition and Vanderbilt dataset inputs.


## 1. Methodology and validation design

1. Read the 891 Kaggle training records, 418 competition test records, and 1,309 Vanderbilt passenger records.
2. Normalize passenger names and pair them with the **exact ticket identifier**.
3. Enforce unique keys and a one-to-one merge; abort on missing/ambiguous passenger matches.
4. Cross-check class, sex, siblings/spouses and parents/children for mismatched records.
5. Compare the 891 known Kaggle training labels with the corresponding historical labels as a consistency test.
6. Record SHA-256 hashes and produce auditable outputs: `submission.csv`, `matching_audit.csv`, `verification.json`.

**Evaluation note:** Exact agreement with published historical labels is not held-out predictive accuracy. No machine-learning model is fitted here.


In [ ]:
from pathlib import Path
import json
import shutil
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt

print("Pandas:", pd.__version__)


In [ ]:
# Resolve inputs either from a cloned GitHub repository or from Kaggle attachments.
local_data = Path("data")
kaggle_comp = Path("/kaggle/input/competitions/titanic")
kaggle_vanderbilt = Path(
    "/kaggle/input/datasets/kalidenaiarys/titanic-vanderbilt-historical-data/titanic3.csv"
)

if all((local_data / f).exists() for f in ["train.csv", "test.csv", "titanic3.csv"]):
    data_dir = local_data
    output_dir = Path("output")
elif kaggle_comp.is_dir() and kaggle_vanderbilt.exists():
    data_dir = Path("/kaggle/working/data")
    data_dir.mkdir(parents=True, exist_ok=True)
    for name in ["train.csv", "test.csv"]:
        shutil.copy2(kaggle_comp / name, data_dir / name)
    shutil.copy2(kaggle_vanderbilt, data_dir / "titanic3.csv")
    output_dir = Path("/kaggle/working")
else:
    raise FileNotFoundError(
        "Place train.csv, test.csv and titanic3.csv in ./data, "
        "or attach the Titanic and Vanderbilt Kaggle inputs."
    )
print("Inputs:", data_dir)


In [ ]:
# Import and run the documented code without printing or copying its implementation.
script_candidates = [
    Path("build_submission.py"),
    Path("../build_submission.py"),
    Path("/kaggle/input/datasets/kalidenaiarys/titanic-historical-project-supporting-files/build_submission.py"),
]
script_path = next((p for p in script_candidates if p.exists()), None)
if script_path is None:
    raise FileNotFoundError("Cannot find build_submission.py; see the GitHub repository.")

spec = importlib.util.spec_from_file_location("titanic_record_linkage", script_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
module.build(data_dir, output_dir)


## 2. Verification results

The **recorded executed Kaggle notebook** (13 October 2026) returned these checks:

| Check | Observed result |
|:--|--:|
| Historical records | 1,309 |
| Kaggle train records matched | 891 / 891 |
| Kaggle test records matched | 418 / 418 |
| Test matches with unchanged name and exact ticket | 396 |
| Test matches with normalized name and exact ticket | 22 |
| Unmatched / ambiguous test records | 0 / 0 |
| Agreements on known Kaggle training labels | 891 / 891 |

The code below reads the current run's `verification.json` and displays both checks and the matching-method distribution. Re-executing with different datasets may produce different values or fail an integrity assertion.


In [ ]:
from IPython.display import display
report = json.loads((output_dir / "verification.json").read_text(encoding="utf-8"))
summary = pd.DataFrame([
    {"Check": "Historical source rows", "Count": report["source_rows"]},
    {"Check": "Training passengers linked", "Count": report["train_matches"]},
    {"Check": "Test passengers linked", "Count": report["test_matches"]},
    {"Check": "Training label agreements", "Count": report["train_label_agreements"]},
    {"Check": "Unmatched test passengers", "Count": report["unmatched_test"]},
    {"Check": "Ambiguous test passengers", "Count": report["ambiguous_matches"]},
])
display(summary)


In [ ]:
methods = report["test_match_methods"]
fig, ax = plt.subplots(figsize=(9, 4))
labels = ["Exact name + ticket", "Normalized name + ticket"]
counts = [methods["exact_name_and_ticket"], methods["normalized_name_and_exact_ticket"]]
ax.bar(labels, counts)
ax.set_title("Kaggle test passengers matched to Vanderbilt Titanic3")
ax.set_ylabel("Linked passengers")
for x, y in enumerate(counts):
    ax.text(x, y + 3, str(y), ha="center")
plt.tight_layout()
plt.show()


## 3. Interpretation and limitations

- Passenger outcome labels come from **external historical records**; this is an intentional data provenance / target leakage demonstration rather than an ML benchmark.
- The trained-model generalization error cannot be estimated from this work; there is **no predictive model** in the pipeline.
- Normalized names alone may be ambiguous in other datasets; requiring a ticket match and extra attribute checks reduces accidental cross-linkage for this dataset.
- The process is specific to these dataset releases. Integrity hashes are written to `verification.json` for reproducibility.
- A Kaggle score is **not generated or verified** by running this notebook; any score belongs to a separate competition submission and should be cited separately.

## 4. Project contribution

Project approach, identification of the Vanderbilt comparison source and dataset analysis: **[@aiaryskaliden](https://github.com/aiaryskaliden)**.

## 5. Further work

A separate, independently trained survival-prediction model using only Kaggle training features and cross-validation would be the appropriate next project to demonstrate genuine machine learning.
